In [1]:
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model

2026-07-13 13:16:39.912160: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-13 13:16:39.930720: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-13 13:16:39.930738: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-13 13:16:39.931300: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-13 13:16:39.934808: I tensorflow/core/platform/cpu_feature_guar

In [2]:
with h5py.File('processed_physics_data.h5', 'r') as f:
    X_train = np.expand_dims(f['X_train'][:], -1)
    y_train = f['y_train'][:]
    X_val = np.expand_dims(f['X_val'][:], -1)
    y_val = f['y_val'][:]

In [3]:
num_train, h, w, _ = X_train.shape
grid_x, grid_y = np.meshgrid(np.linspace(-1, 1, w), np.linspace(-1, 1, h))
grid_x = np.tile(grid_x[np.newaxis, ..., np.newaxis], (num_train, 1, 1, 1))
grid_y = np.tile(grid_y[np.newaxis, ..., np.newaxis], (num_train, 1, 1, 1))
X_train_cc = np.concatenate([X_train, grid_x, grid_y], axis=-1)

In [12]:
num_val = X_val.shape[0]
grid_x_v, grid_y_v = np.meshgrid(np.linspace(-1, 1, w), np.linspace(-1, 1, h))
grid_x_v = np.tile(grid_x_v[np.newaxis, ..., np.newaxis], (num_val, 1, 1, 1))
grid_y_v = np.tile(grid_y_v[np.newaxis, ..., np.newaxis], (num_val, 1, 1, 1))
X_val_cc = np.concatenate([X_val, grid_x_v, grid_y_v], axis=-1)

In [13]:
inputs = Input(shape=(24, 36, 3))
x = Conv2D(64, (3, 3), padding='same')(inputs)
x = BatchNormalization()(x)
x = Activation('relu')(x)

In [14]:
shortcut = Conv2D(64, (1, 1), padding='same')(x)
x = Conv2D(64, (3, 3), padding='same')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Add()([x, shortcut])

In [15]:
x = MaxPooling2D((2, 2))(x)
x = Conv2D(128, (3, 3), padding='same')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)

In [16]:
shortcut = Conv2D(128, (1, 1), padding='same')(x)
x = Conv2D(128, (3, 3), padding='same')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = Add()([x, shortcut])

In [17]:
x = MaxPooling2D((2, 2))(x)
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

In [18]:
outputs = Dense(1, activation='sigmoid')(x)
resnet_cc_model = Model(inputs, outputs)
fl = tf.keras.losses.BinaryFocalCrossentropy(gamma=2.5, alpha=0.85)
resnet_cc_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=fl, metrics=[tf.keras.metrics.AUC(name='auc')])

In [19]:
with tf.device('/CPU:0'):
    train_ds = tf.data.Dataset.from_tensor_slices((X_train_cc, y_train))
    val_ds = tf.data.Dataset.from_tensor_slices((X_val_cc, y_val))
train_ds = train_ds.shuffle(1024).batch(32).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(32).prefetch(tf.data.AUTOTUNE)

In [20]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint('resnet_cc_best.h5', monitor='val_auc', save_best_only=True, mode='max'),
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=12, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=3, min_lr=1e-6)
]

In [21]:
history = resnet_cc_model.fit(
    train_ds, validation_data=val_ds,
    epochs=200, callbacks=callbacks, verbose=1
)

Epoch 1/200


2026-07-13 13:18:22.945197: I external/local_xla/xla/service/service.cc:168] XLA service 0x5c7ec2a71050 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-07-13 13:18:22.945218: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce GT 1030, Compute Capability 6.1
2026-07-13 13:18:22.979582: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-07-13 13:18:23.200687: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8902


   3/2744 ━━━━━━━━━━━━━━━━━━━━ 2:50 62ms/step - auc: 0.6383 - loss: 1.7932 

I0000 00:00:1783916315.201487 3418034 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2744/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - auc: 0.6739 - loss: 0.1852

...

2744/2744 ━━━━━━━━━━━━━━━━━━━━ 181s 66ms/step - auc: 0.8276 - loss: 0.0485 - val_auc: 0.8645 - val_loss: 0.0439 - learning_rate: 0.0010
Epoch 3/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - auc: 0.8220 - loss: 0.0500

2744/2744 ━━━━━━━━━━━━━━━━━━━━ 182s 66ms/step - auc: 0.8290 - loss: 0.0481 - val_auc: 0.8680 - val_loss: 0.0429 - learning_rate: 0.0010
Epoch 4/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 181s 66ms/step - auc: 0.8421 - loss: 0.0467 - val_auc: 0.8676 - val_loss: 0.0428 - learning_rate: 0.0010
Epoch 5/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - auc: 0.8367 - loss: 0.0487

2744/2744 ━━━━━━━━━━━━━━━━━━━━ 182s 66ms/step - auc: 0.8391 - loss: 0.0473 - val_auc: 0.8693 - val_loss: 0.0449 - learning_rate: 0.0010
Epoch 6/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 181s 66ms/step - auc: 0.8479 - loss: 0.0459 - val_auc: 0.8681 - val_loss: 0.0429 - learning_rate: 0.0010
Epoch 7/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 182s 66ms/step - auc: 0.8490 - loss: 0.0460 - val_auc: 0.8680 - val_loss: 0.0473 - learning_rate: 0.0010
Epoch 8/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 181s 66ms/step - auc: 0.8511 - loss: 0.0455 - val_auc: 0.8678 - val_loss: 0.0426 - learning_rate: 0.0010
Epoch 9/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - auc: 0.8597 - loss: 0.0452

2744/2744 ━━━━━━━━━━━━━━━━━━━━ 182s 66ms/step - auc: 0.8601 - loss: 0.0442 - val_auc: 0.8710 - val_loss: 0.0425 - learning_rate: 5.0000e-04
Epoch 10/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 182s 66ms/step - auc: 0.8606 - loss: 0.0440 - val_auc: 0.8696 - val_loss: 0.0430 - learning_rate: 5.0000e-04
Epoch 11/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 182s 66ms/step - auc: 0.8628 - loss: 0.0436 - val_auc: 0.8708 - val_loss: 0.0431 - learning_rate: 5.0000e-04
Epoch 12/200
2743/2744 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - auc: 0.8635 - loss: 0.0445

2744/2744 ━━━━━━━━━━━━━━━━━━━━ 181s 66ms/step - auc: 0.8647 - loss: 0.0433 - val_auc: 0.8711 - val_loss: 0.0436 - learning_rate: 5.0000e-04
Epoch 13/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 181s 66ms/step - auc: 0.8663 - loss: 0.0431 - val_auc: 0.8697 - val_loss: 0.0429 - learning_rate: 5.0000e-04
Epoch 14/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 181s 66ms/step - auc: 0.8689 - loss: 0.0426 - val_auc: 0.8687 - val_loss: 0.0435 - learning_rate: 5.0000e-04
Epoch 15/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 176s 64ms/step - auc: 0.8706 - loss: 0.0423 - val_auc: 0.8674 - val_loss: 0.0449 - learning_rate: 5.0000e-04
Epoch 16/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 173s 63ms/step - auc: 0.8776 - loss: 0.0411 - val_auc: 0.8676 - val_loss: 0.0448 - learning_rate: 2.5000e-04
Epoch 17/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 181s 66ms/step - auc: 0.8804 - loss: 0.0404 - val_auc: 0.8673 - val_loss: 0.0443 - learning_rate: 2.5000e-04
Epoch 18/200
2744/2744 ━━━━━━━━━━━━━━━━━━━━ 182s 66ms/step - auc: 0.8822 - loss: 0.0401 - val_a